# binderscope — walkthrough

Ranking and dashboard stages on the bundled **synthetic** sample data.
No PyRosetta, no GPU and no design campaign required.

> The metrics in `examples/demo_metrics.csv` are randomly generated with
> realistic correlations. They carry no biological meaning.

For a real run, see `configs/pdl1_example.yaml` and the `binderscope score`
command, which needs PyRosetta and BindCraft output.


## 1 · Load a run configuration

Everything target-specific lives in the YAML file: chains, residue mappings,
thresholds and ranking weights. The code carries no knowledge of any target.


In [ ]:
from pathlib import Path

import pandas as pd

from binderscope import load_config

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = load_config(ROOT / 'configs' / 'demo.yaml')

print(f'run:            {config.name}')
print(f'binder chain:   {config.binder_chain}')
print(f'rank metrics:   {len(config.rank_metrics)}')
print(f'max score:      {config.total_weight:g}')
print(f'needs reversion: {config.needs_reversion}')


## 2 · The metrics table

In a real run this is written by `binderscope score` and `binderscope merge`:
PyRosetta interface metrics, placement geometry if a reference is configured,
and the AlphaFold2 columns BindCraft already produced.


In [ ]:
df = pd.read_csv(ROOT / 'examples' / 'demo_metrics.csv')
print(f'{len(df)} designs × {len(df.columns)} columns')
df[['design', 'dG', 'dSASA', 'sc', 'packstat', 'unsat_hbonds', 'Average_i_pTM']].head()


## 3 · Hard filters

Thresholds come from the config, so the same run is reproducible and the
numbers are reviewable without reading any code.


In [ ]:
from binderscope.ranking import apply_filters

passing = apply_filters(df, config.filters)
print(f'{passing.sum()} of {len(df)} designs pass every enabled filter')

for flt in config.filters:
    state = 'on ' if flt.enabled else 'off'
    print(f'  [{state}] {flt.col} {flt.op} {flt.value:g}')


## 4 · Multi-criteria ranking

Each metric is min-max normalised in its own good direction, multiplied by
its weight and summed. Rows excluded by `ranking.exclude` are dropped
*before* normalisation, so one artefact cannot stretch the scale and
compress every real design into a narrow band.


In [ ]:
from binderscope.ranking import rank_designs

ranked, contributions = rank_designs(df, config.rank_metrics, config.rank_exclude)
print(f'{len(ranked)} designs ranked ({len(df) - len(ranked)} excluded as artefacts)')

columns = ['rank', 'design', 'score'] + [m.col for m in config.rank_metrics]
ranked[columns].head(10).round(3)


### Why a design ranks where it does

The total alone hides the trade-offs. The per-metric contributions show
whether a candidate wins on interface energy, on predictor confidence, or by
being merely adequate everywhere — which is the part that decides what you
would actually order.


In [ ]:
top = ranked.head(8)
breakdown = contributions.head(8).round(2)
breakdown.index = top['design'].str.replace('demo_', '', regex=False)
breakdown


## 5 · The dashboard

One self-contained HTML file with the data inlined: no server, nothing to
install, and it survives being emailed or dropped into a supplement.
The context panel is generated from the config, so the page documents its
own provenance.


In [ ]:
from binderscope.dashboard import build_dashboard

output = build_dashboard(df, config)
print(f'{output}  ({output.stat().st_size / 1024:.0f} KB)')


In [ ]:
from IPython.display import IFrame

IFrame(src=output.relative_to(Path.cwd()) if output.is_relative_to(Path.cwd()) else str(output),
       width='100%', height=700)


---

## Running this on real data

```bash
# optional: rebuild the wild-type target if it was engineered
binderscope revert    my_run.yaml --sessions

# the slow stage — hours of FastRelax, resumable if interrupted
binderscope score     my_run.yaml

binderscope merge     my_run.yaml
binderscope rank      my_run.yaml --contributions
binderscope dashboard my_run.yaml
```

Or `binderscope run my_run.yaml` for all of it in sequence.
